In [2]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/diamonds.db"
r = requests.get(url)

with open("diamonds.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("diamonds.db")

query = """
SELECT
  O.carat,
  O.price,
  O.depth,
  "O"."table",
  O.x,
  O.y,
  O.z,
  C.cut,
  Co.color,
  Cl.clarity
FROM
  Observation AS O
JOIN
  Cut AS C ON O.cut_id = C.cut_id
JOIN
  Color AS Co ON O.color_id = Co.color_id
JOIN
  Clarity AS Cl ON O.clarity_id = Cl.clarity_id
"""

df = pd.read_sql_query(query, conn)
df.head()

,carat,price,depth,table,x,y,z,cut,color,clarity
0,0.23,326,61.5,55.0,3.95,3.98,2.43,Ideal,E,SI2
1,0.21,326,59.8,61.0,3.89,3.84,2.31,Premium,E,SI1
2,0.23,327,56.9,65.0,4.05,4.07,2.31,Good,E,VS1
3,0.29,334,62.4,58.0,4.20,4.23,2.63,Premium,I,VS2
4,0.31,335,63.3,58.0,4.34,4.35,2.75,Good,J,SI2


## 📋 Recap del Análisis Exploratorio

En el notebook anterior exploramos el dataset de **53,940 diamantes**. Aquí un resumen rápido:

### Variables
| Tipo | Variables |
|---|---|
| **Numéricas** | `carat`, `depth`, `table`, `x`, `y`, `z`, `price` |
| **Categóricas (ordinales)** | `cut` (Fair → Premium), `color` (D → J), `clarity` (IF → I1) |

### Hallazgos clave
- 💎 **`carat`**, **`x`**, **`y`** y **`z`** tienen la correlación más fuerte con `price`
- ⚠️ `depth` y `table` muestran alta dispersión y outliers elevados
- 📊 `price` y `carat` tienen distribuciones **sesgadas a la derecha**
- 🔗 Las variables de dimensión (`x`, `y`, `z`) están **altamente correlacionadas entre sí** → posible multicolinealidad

### Variable objetivo
> Predeciremos **`price`** (precio en USD) usando las demás características del diamante.

## 1️⃣ Preprocesamiento

Antes de ajustar el modelo, preparamos los datos:
- Separamos la variable objetivo (`price`) de las variables predictoras
- Dividimos en **80% entrenamiento / 20% prueba**

In [3]:
# Preprocesar 1
from sklearn.model_selection import train_test_split

X= df.drop(columns=["price"])
y= df["price"]

X_train, X_test, y_train, y_test= train_test_split(
    X, y, test_size= 0.2, random_state= 42
)

print(f"Train: {X_train.shape} | Test:{X_test.shape}")

Train: (43152, 9) | Test:(10788, 9)


### Encoding de variables categóricas

Las variables `cut`, `color` y `clarity` son categóricas ordinales.  
Usamos **OrdinalEncoder** respetando el orden natural de cada una.

In [13]:
# Preprocesar 2

from sklearn.preprocessing import OrdinalEncoder

#definir orden natural de cada variable categórica ordinal 

cut_order=['Fair','Good','Very Good','Premium','Ideal']
color_order=['J','I','H','G','F','E','D']
clarity_order=['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF']

enc=OrdinalEncoder(
    categories=[cut_order,color_order,clarity_order]
)
cat_cols=['cut','color','clarity']

#fit en conjunto de entrenamiento, transofrm() en ambos
X_train_enc=X_train.copy()
X_test_enc=X_test.copy()

X_train_enc[cat_cols]=enc.fit_transform(X_train[cat_cols])
X_test_enc[cat_cols]=enc.transform(X_test[cat_cols])

X_train_enc.head()

,carat,depth,table,x,y,z,cut,color,clarity
26546,2.01,58.1,64.0,8.23,8.19,4.77,1.0,4.0,1.0
9159,1.01,60.0,60.0,6.57,6.49,3.92,2.0,5.0,1.0
14131,1.10,62.5,58.0,6.59,6.54,4.10,3.0,2.0,3.0
15757,1.50,61.5,65.0,7.21,7.17,4.42,1.0,5.0,1.0
24632,1.52,62.1,57.0,7.27,7.32,4.53,2.0,3.0,4.0


## 2️⃣ Ajuste del Modelo — Regresión Lineal (OLS)

Ajustamos un modelo **crudo** con todas las variables, sin ningún tipo de selección.  
Usamos `statsmodels` para ver el resumen estadístico completo.

In [15]:
# Ajustar Regresión
import statsmodels.api as sm

X_train_const=sm.add_constant(X_train_enc)

model_sm=sm.OLS(y_train, X_train_const)
results=model_sm.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.907
Model:                            OLS   Adj. R-squared:                  0.907
Method:                 Least Squares   F-statistic:                 4.694e+04
Date:                Fri, 06 Mar 2026   Prob (F-statistic):               0.00
Time:                        01:15:56   Log-Likelihood:            -3.6770e+05
No. Observations:               43152   AIC:                         7.354e+05
Df Residuals:                   43142   BIC:                         7.355e+05
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       3830.8354    472.330      8.110      0.0

## 3️⃣ Evaluación en Test

Medimos el desempeño del modelo con datos que **nunca vio durante el entrenamiento**.

In [16]:
# Evaluar Regresión
from sklearn.metrics import r2_score,mean_squared_error
from math import sqrt

X_test_const= sm.add_constant(X_test_enc)
y_pred= results.predict(X_test_const)

r2= r2_score(y_test, y_pred)
rmse= sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 en test: {r2:.4f}")
print(f"RMSE en test: {rmse:.2f} USD")

R2 en test: 0.9057
RMSE en test: 1224.60 USD


In [18]:
df['price'].describe()

count    53940.000000
mean      3932.799722
std       3989.439738
min        326.000000
25%        950.000000
50%       2401.000000
75%       5324.250000
max      18823.000000
Name: price, dtype: float64

In [20]:
import numpy as np

#normalizar el precio 
y_train_log=np.log(y_train)
y_test_log=np.log(y_test)

#normalizar carat
X_train_log=X_train_enc.copy()
X_test_log=X_test_enc.copy()

X_train_log['carat']=np.log(X_train_enc['carat'])
X_test_log['carat']=np.log(X_test_enc['carat'])

#ajustar de nuevo 

X_train_log_const=sm.add_constant(X_train_log)
X_test_log_const=sm.add_constant(X_test_log)

model_log=sm.OLS(y_train_log,X_train_log_const)
results_log=model_log.fit()

print(results_log.summary())


                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.979
Model:                            OLS   Adj. R-squared:                  0.979
Method:                 Least Squares   F-statistic:                 2.272e+05
Date:                Fri, 06 Mar 2026   Prob (F-statistic):               0.00
Time:                        02:06:57   Log-Likelihood:                 21860.
No. Observations:               43152   AIC:                        -4.370e+04
Df Residuals:                   43142   BIC:                        -4.361e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          7.9246      0.070    112.509      0.0

In [21]:
y_pred_log=results_log.predict(X_test_log_const)

r2_log=r2_score(y_test_log,y_pred_log)

#revertir logaritmo

y_pred_usd=np.exp(y_pred_log)
rmse_usd=sqrt(mean_squared_error(y_test,y_pred_usd))

print(f"R2:{r2_log:.4f}")
print(f"RMSE:{rmse_usd:.2f}USD")

R2:0.9789
RMSE:1026.37USD
